# 리포트 50 — 5G SSB 는 걷는 드론에서 접힌다

> ### 한 일
> **세 기준신호의 **물리** 반복률에서 무모호 도플러를 계산하고, 기준 표적 속도의 참 도플러가 어디로 접히는지를 같은 표에 적었다.**

### 결과
1. SSB 의 물리 반복률은 50 Hz [^1] 이고 무모호 속도가 1.07 m/s [^2] 다 — 걷는 속도의 드론이 그 위에 있다.
2. 기준 표적 속도에서 참 도플러 64.0 Hz [^3] 가 14.0 Hz [^4] 로 접힌다.
3. 같은 조건에서 WiFi 는 무모호 속도 14.4 m/s [^5], LTE 는 40.7 m/s [^6] 로 참 도플러를 그대로 유지한다 — LTE 값은 규격 주기에서 오고, WiFi 값은 혼잡 AP 트래픽 가정에서 온 선언값이다(`src/waveforms.py:112`).
4. 접힘을 정하는 것은 물리 반복률 하나다 — CPI 는 도플러 가드 폭을 정하고, 모호속도는 표본화율의 성질이라 CPI 로 안 움직인다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 물리 반복률 | **출처가 갈린다** — LTE CRS 는 `TS 36.211` 의 매 서브프레임 송신이, 5G SSB 는 `TS 38.213` 의 기본 버스트 주기가 고정한 규격값이다. **WiFi 는 트래픽 가정에서 온 선언값**이다 — `src/waveforms.py:112` 의 `PILOT_RATE_HZ` 가 혼잡 AP 대표값 1000 Hz [^7] 를 든다. 셋 다 검출기 프레임률과 다른 양이라 이름을 갈라 싣는다 |
| 접힘 계산 | 무모호 도플러 ±PRF/2 를 넘는 참 도플러를 그 구간으로 되접어 아래 표에 적는다 |
| 기준 표적 속도 | 이 프로젝트의 기준 표적 속도 하나에서 계산한다 — 속도를 바꾸면 접힌 자리도 바뀐다 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/verify_ambiguity.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part08_illuminators.py
```

| | |
|---|---|
| 출력 | `outputs/verify_ambiguity.json` |
| 소요 | ② GPU 1장 수 분 · ④ CPU 20초 안쪽 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [편 45 «5G 는 좁고 드물다»](45_5g-double-cost.ipynb) | 5G 가 거리·속도 두 축에서 무는 대가 |
| [편 49 «검출기가 실제로 쓰는 커널 그대로 모호함수를…»](49_ambiguity.ipynb) | 검출기 커널이 만드는 응답의 모양 |

---

## 물리 반복률이 무모호 속도를 정한다

| 기준신호 | 물리 PRF | 그 값의 판 | 무모호 속도 | 접히는가 |
|---|---|---|---|---|
| WiFi VHT-LTF | 1000 Hz [^7] | **선언값** — 혼잡 AP 트래픽 가정(`src/waveforms.py:112`) | 14.4 m/s [^5] | 아니오 [^8] |
| LTE CRS | 1000 Hz [^9] | 규격값 — `TS 36.211`, 매 서브프레임 | 40.7 m/s [^6] | 아니오 [^10] |
| 5G SSB | 50 Hz [^1] | 규격값 — `TS 38.213`, 기본 버스트 주기 | 1.07 m/s [^2] | 예 [^11] |

WiFi 행의 값은 트래픽 시나리오가 정한다 — 비콘만 뜨는 유휴 AP 에서는 그 반복률이 SSB 아래로 내려간다(`src/waveforms.py:105`). 이 편의 WiFi 결론은 혼잡 AP 가정 위에 선다.

## 접힌 자리는 어디인가

기준 표적 속도에서 5G 의 참 도플러 64.0 Hz [^3] 가 무모호 구간 ±25 Hz [^12] 안으로 되접혀 14.0 Hz [^4] 에 나타난다.

접힌 표적은 사라지는 것이 아니라 **엉뚱한 속도로 보고된다**. 그래서 이 대가는 감도가 아니라 판정의 정합성에 든다.

같은 접힘이 마이크로도플러 축에도 그대로 온다 — 검출이 죽은 칸에서 빗살이 어디까지 남는지, 그리고 5G 에서 그 빗살이 어떻게 접히는지는 [리포트 11-2 «기준채널이 현실이면 얼마를 잃는가»](11_2_two_channel.ipynb) 의 마지막 절이 같은 사슬에서 잰다.

## 두 배의 대가의 나머지 절반

5G 는 좁아서 거리 눈금이 거칠고, 드물어서 속도 눈금이 접힌다 — [편 45 «5G 는 좁고 드물다 — 두 배의 대가를 치른다»](45_5g-double-cost.ipynb) 가 든 두 축의 뒤쪽이 여기다.

접힘을 정하는 것은 물리 반복률 50 Hz [^1] 하나이고, CPI 가 정하는 것은 도플러 가드 폭이다. 그 CPI 스윕은 [편 62 «CPI 를 늘리면 세 파형 모두 블라인드율이…»](62_cpi-sweep.ipynb) 가 싣고, CPI 로도 안 움직이는 잔여분은 [편 63 «모호속도는 표본화율의 성질이라 CPI 와 무관…»](63_cpi-residual.ipynb) 가 든다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 검출기 CPI 24 ms [^13] 를 스윕해 SSB 도플러 가드 폭을 PRF 대비로 잰다 | 5G 상시 기준신호의 접힘이 단일 CPI 결과인지 체제인지가 수치로 갈린다 | `outputs/cpi_guard_sweep.json` → [편 62 «CPI 를 늘리면 세 파형 모두 블라인드율이…»](62_cpi-sweep.ipynb) |
| 표적 속도를 격자로 넓혀 접히는 속도 구간을 지도로 만든다 | 어느 속도대가 5G 에서 엉뚱한 속도로 보고되는지가 확정된다 | `benchmark/verify_ambiguity.py:108` |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 13개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.physical.prf_physical_hz` | 50 |
| [^2] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.physical.v_unamb_phys_ms` | 1.071 |
| [^3] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.physical.fd_true_hz` | 64.03 |
| [^4] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.physical.fd_aliased_phys_hz` | 14.03 |
| [^5] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.physical.v_unamb_phys_ms` | 14.39 |
| [^6] | `outputs/verify_ambiguity.json` | `waveforms.lte_G1.physical.v_unamb_phys_ms` | 40.67 |
| [^7] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.physical.prf_physical_hz` | 1000 |
| [^8] | `outputs/verify_ambiguity.json` | `waveforms.wifi_G1.physical.aliased` | 아니오 |
| [^9] | `outputs/verify_ambiguity.json` | `waveforms.lte_G1.physical.prf_physical_hz` | 1000 |
| [^10] | `outputs/verify_ambiguity.json` | `waveforms.lte_G1.physical.aliased` | 아니오 |
| [^11] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.physical.aliased` | 예 |
| [^12] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.physical.fd_unamb_phys_hz` | 25 |
| [^13] | `outputs/verify_ambiguity.json` | `waveforms.nr_G1.physical.cpi_model_ms` | 24 |